# Transformer Encoder–Decoder: Penerjemahan Inggris → Spanyol

**Buku:** *Deep Learning dengan Python* — Bab 8, Atensi dan Transformer

Arsitektur lengkap "Attention Is All You Need" versi mini: encoder, decoder dengan masker kausal dan *cross-attention*, dilatih pada ±118 ribu pasangan kalimat.

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+. Notebook ini memakai backend **TensorFlow** (dibutuhkan oleh `tf.data`/`tf.GradientTape`/ekspor).

**Perkiraan waktu (GPU T4):** ±30 menit untuk 20 epoch

**Isi notebook:**
- Penyiapan data teks berpasangan dengan `TextVectorization` dan `tf.data`.
- Decoder dengan *masked self-attention* + *cross-attention* ke keluaran encoder.
- *Teacher forcing* saat latih; *greedy decoding* token demi token saat inferensi.
- Loss yang mengabaikan token padding (`ignore_class=0`).

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())
import tensorflow as tf

## 1. Data

In [ ]:
import zipfile, random, re, string
import numpy as np

jalur = keras.utils.get_file("spa-eng.zip",
                             origin="http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip")
with zipfile.ZipFile(jalur) as zf:
    baris = zf.read("spa-eng/spa.txt").decode("utf-8").strip().split("\n")

pasangan = []
for b in baris:
    eng, spa = b.split("\t")
    pasangan.append((eng, "[start] " + spa + " [end]"))      # token awal/akhir untuk decoder

random.seed(42); random.shuffle(pasangan)
n_val = int(0.15 * len(pasangan))
latih = pasangan[:-2 * n_val]
val = pasangan[-2 * n_val:-n_val]
uji = pasangan[-n_val:]
print(len(latih), len(val), len(uji)); print(random.choice(latih))

In [ ]:
from keras import layers

KOSAKATA, PANJANG, BATCH = 15000, 20, 64

hapus = (string.punctuation + "¿").replace("[", "").replace("]", "")   # [start]/[end] dipertahankan
def standarisasi(s):
    return tf.strings.regex_replace(tf.strings.lower(s), f"[{re.escape(hapus)}]", "")

vek_eng = layers.TextVectorization(max_tokens=KOSAKATA, output_sequence_length=PANJANG)
vek_spa = layers.TextVectorization(max_tokens=KOSAKATA, output_sequence_length=PANJANG + 1,
                                   standardize=standarisasi)
vek_eng.adapt([p[0] for p in latih])
vek_spa.adapt([p[1] for p in latih])

def format_ds(eng, spa):
    eng, spa = vek_eng(eng), vek_spa(spa)
    # masukan decoder: kalimat target tanpa token terakhir; target: digeser satu langkah
    return {"enc": eng, "dec": spa[:, :-1]}, spa[:, 1:]

def buat_ds(data):
    eng, spa = zip(*data)
    ds = tf.data.Dataset.from_tensor_slices((list(eng), list(spa))).batch(BATCH)
    return ds.map(format_ds, num_parallel_calls=tf.data.AUTOTUNE).cache().prefetch(tf.data.AUTOTUNE)

ds_latih, ds_val = buat_ds(latih).shuffle(2048), buat_ds(val)
for x, y in ds_latih.take(1):
    print({k: v.shape for k, v in x.items()}, y.shape)

## 2. Lapisan encoder dan decoder

In [ ]:
from keras import layers, ops

class EmbeddingPosisi(layers.Layer):
    """Embedding token + embedding posisi yang dipelajari."""
    def __init__(self, panjang, kosakata, dim, **kwargs):
        super().__init__(**kwargs)
        self.token = layers.Embedding(kosakata, dim)
        self.posisi = layers.Embedding(panjang, dim)
        self.cfg = dict(panjang=panjang, kosakata=kosakata, dim=dim)

    def call(self, x):
        pos = ops.arange(0, ops.shape(x)[-1])
        return self.token(x) + self.posisi(pos)

    def get_config(self):
        return {**super().get_config(), **self.cfg}


class EncoderTransformer(layers.Layer):
    """Satu blok encoder: self-attention + FFN, masing-masing dengan residual dan LayerNorm."""
    def __init__(self, dim, dim_ff, n_head, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.atensi = layers.MultiHeadAttention(num_heads=n_head, key_dim=dim // n_head)
        self.ffn = keras.Sequential([layers.Dense(dim_ff, activation="gelu"), layers.Dense(dim)])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)
        self.cfg = dict(dim=dim, dim_ff=dim_ff, n_head=n_head, dropout=dropout)

    def call(self, x, token=None, training=False):
        mask = None
        if token is not None:                                  # abaikan posisi padding (token 0)
            mask = ops.expand_dims(ops.not_equal(token, 0), 1)
        a = self.atensi(x, x, attention_mask=mask, training=training)
        x = self.norm1(x + self.drop1(a, training=training))
        return self.norm2(x + self.drop2(self.ffn(x), training=training))

    def get_config(self):
        return {**super().get_config(), **self.cfg}


class DecoderTransformer(layers.Layer):
    """Masked self-attention -> cross-attention ke encoder -> FFN."""
    def __init__(self, dim, dim_ff, n_head, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.atensi_diri = layers.MultiHeadAttention(num_heads=n_head, key_dim=dim // n_head)
        self.atensi_silang = layers.MultiHeadAttention(num_heads=n_head, key_dim=dim // n_head)
        self.ffn = keras.Sequential([layers.Dense(dim_ff, activation="gelu"), layers.Dense(dim)])
        self.norm = [layers.LayerNormalization(epsilon=1e-6) for _ in range(3)]
        self.drop = [layers.Dropout(dropout) for _ in range(3)]
        self.cfg = dict(dim=dim, dim_ff=dim_ff, n_head=n_head, dropout=dropout)

    def call(self, x, token_dec, konteks, token_enc, training=False):
        mask_dec = ops.expand_dims(ops.not_equal(token_dec, 0), 1)
        mask_enc = ops.expand_dims(ops.not_equal(token_enc, 0), 1)
        a = self.atensi_diri(x, x, attention_mask=mask_dec, use_causal_mask=True, training=training)
        x = self.norm[0](x + self.drop[0](a, training=training))
        a = self.atensi_silang(query=x, value=konteks, key=konteks, attention_mask=mask_enc, training=training)
        x = self.norm[1](x + self.drop[1](a, training=training))
        return self.norm[2](x + self.drop[2](self.ffn(x), training=training))

    def get_config(self):
        return {**super().get_config(), **self.cfg}

## 3. Model lengkap

In [ ]:
DIM, DIM_FF, N_HEAD = 256, 2048, 8

token_enc = keras.Input(shape=(None,), dtype="int64", name="enc")
token_dec = keras.Input(shape=(None,), dtype="int64", name="dec")

x = EmbeddingPosisi(PANJANG, KOSAKATA, DIM)(token_enc)
konteks = EncoderTransformer(DIM, DIM_FF, N_HEAD)(x, token_enc)

y = EmbeddingPosisi(PANJANG, KOSAKATA, DIM)(token_dec)
y = DecoderTransformer(DIM, DIM_FF, N_HEAD)(y, token_dec, konteks, token_enc)
y = layers.Dropout(0.5)(y)
keluaran = layers.Dense(KOSAKATA, activation="softmax")(y)

transformer = keras.Model([token_enc, token_dec], keluaran)
transformer.summary()

In [ ]:
transformer.compile(optimizer="rmsprop",
                    loss=keras.losses.SparseCategoricalCrossentropy(ignore_class=0),
                    metrics=["accuracy"])
transformer.fit(ds_latih, epochs=20, validation_data=ds_val)

## 4. Menerjemahkan (*greedy decoding*)
Decoder dijalankan berulang: setiap langkah menambah satu token hasil prediksi, sampai muncul `[end]`.

In [ ]:
kosakata_spa = vek_spa.get_vocabulary()

def terjemahkan(kalimat):
    enc = vek_eng([kalimat])
    hasil = "[start]"
    for i in range(PANJANG):
        dec = vek_spa([hasil])[:, :-1]
        prediksi = keras.ops.convert_to_numpy(transformer({"enc": enc, "dec": dec}, training=False))
        kata = kosakata_spa[int(np.argmax(prediksi[0, i]))]
        hasil += " " + kata
        if kata == "[end]":
            break
    return hasil.replace("[start] ", "").replace(" [end]", "")

for eng, spa in random.sample(uji, 8):
    print("EN :", eng)
    print("REF:", spa.replace("[start] ", "").replace(" [end]", ""))
    print("MT :", terjemahkan(eng)); print()

**Eksperimen:** terjemahkan kalimat buatan Anda sendiri, misalnya `terjemahkan("I would like a cup of coffee.")`. Coba kalimat yang panjang atau berstruktur tidak biasa, lalu amati di mana model mulai gagal.